## Evaluation & Comparison

Workflow:
1. Summarize best results from binary classification (notebook 04)
2. Summarize best results from multiclass classification (notebook 05)
3. Compare binary vs multiclass performance
4. Discuss feature importance of the best models
5. Final conclusions

In [ ]:
# 1. Summarize best results from binary classification (notebook 04)

import pandas as pd
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

random_state = 42

# Load best model
best_model_bin = joblib.load("../models/best_model_bin.pkl")

# Load test data
X_test_bin = pd.read_parquet("../data/processed/X_test.parquet")
y_test_bin = pd.read_parquet("../data/processed/y_label_test.parquet")['label']

y_pred_bin = best_model_bin.predict(X_test_bin)
f1_binary = f1_score(y_test_bin, y_pred_bin)

print("Best binary model (from notebook 4):")
print(classification_report(y_test_bin, y_pred_bin, zero_division=0))
print(f"F1: {f1_binary:.3f}")

In [ ]:
# 2. Summarize best results from multiclass classification (notebook 05)

# Load best model
best_model_multi = joblib.load("../models/best_model_mlt.pkl")

# Load test data
X_test_multi = pd.read_parquet("../data/processed/X_test.parquet")
y_test_multi = pd.read_parquet("../data/processed/y_cat_test.parquet")['attack_cat']

y_pred_multi = best_model_multi.predict(X_test_multi)
f1_multi = f1_score(y_test_multi, y_pred_multi, average='macro')

print("Best multiclass model (from notebook 5):")
print(classification_report(y_test_multi, y_pred_multi, zero_division=0))
print(f"Macro F1: {f1_multi:.3f}")

In [ ]:
# 3. Compare binary vs multiclass performance

import os
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

#Binary confusion matrix

os.makedirs("../doc/images", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_bin, y_pred_bin, normalize='true', cmap='Blues', ax=ax
)
ax.set_title("Confusion Matrix - Binary Classification\n(0: Normal, 1: Attack)")
plt.tight_layout()
plt.savefig("../doc/images/cm_binary.pdf", format="pdf", bbox_inches="tight")
plt.show()

# Multiclass confusion matrix
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_multi, y_pred_multi, normalize='true', cmap='Reds',
    ax=ax, xticks_rotation=60, values_format='.2f'
)
ax.set_title("Confusion Matrix - Multiclass Classification")
plt.tight_layout()
plt.savefig("../doc/images/cm_multiclass.pdf", format="pdf", bbox_inches="tight")
plt.show()

### Confusion Matrix discussion

The binary model shows a favorable pattern for an intrusion detection context: attack recall (96%) is higher than normal-traffic recall (88%). This means very few real attacks go undetected (approx. 4% false negative rate) at the cost of some false positives (12%). In cybersecurity, this is a reasonable trade-off given the higher cost of missed attacks.

Regarding the multiclass confusion matrix, misclassifications are not random but concentrated among specific category pairs: Backdoor is most often confused with Analysis (41%), DoS with Exploits (32%), and Worms with Exploits (33%) — all minority classes with overlapping traffic characteristics and limited training examples. Normal traffic is also confused with Fuzzers in 13% of cases. Notably, this confusion cannot be explained by class imbalance alone, since Normal is the majority class (over 56,000 examples in the training set). This suggests a genuine overlap between some fuzzing traffic and normal traffic behaviour, supporting the conclusion that the limited macro F1 (0.561) reflects real classification difficulty rather than insufficient training data or a modeling flaw.

In [ ]:
# 4. Discuss feature importance of the best models

import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Binary model feature importance
clf_bin = best_model_bin.best_estimator_.named_steps['classifier']
features_bin = best_model_bin.best_estimator_[:-1].get_feature_names_out(X_test_bin.columns)

importance_bin = pd.Series(
    clf_bin.feature_importances_,
    index = features_bin
).sort_values(ascending=False).head(15)

importance_bin.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title("Top 15 features — Binary (label)")
axes[0].set_xlabel("Importance")

# Multiclass model feature importance
clf_multi = best_model_multi.best_estimator_.named_steps['classifier']
features_multi = best_model_multi.best_estimator_[:-1].get_feature_names_out(X_test_bin.columns)

importance_multi = pd.Series(
    clf_multi.feature_importances_,
    index = features_multi
).sort_values(ascending=False).head(15)

importance_multi.plot(kind='barh', ax=axes[1], color='firebrick')
axes[1].invert_yaxis()
axes[1].set_title("Top 15 features — Multiclass (attack_cat)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("../doc/images/feature_importance.pdf", format="pdf", bbox_inches="tight")
plt.show()

### Feature importance discussion

11 of the top 15 features are shared between the binary and multiclass models,
indicating consistent underlying signal. The main difference is that the binary model
relies more on TCP handshake timing features (ackdat, synack, tcprtt), since anomalous
handshakes are a strong indicator of malicious traffic. The multiclass model instead
relies more heavily on byte-transfer volume features (sbytes, smean, dmean, dbytes),
which is intuitive since different attack categories (e.g. DoS or Exploits) typically
produce different traffic volume patterns, useful to pinpoint the specific attack type
rather than only detecting its presence.

In [ ]:
# 5. Final conclusions

name_bin = type(best_model_bin.best_estimator_[-1]).__name__
name_multi = type(best_model_multi.best_estimator_[-1]).__name__

params_bin = str(best_model_bin.best_params_)
params_multi = str(best_model_multi.best_params_)

summary = pd.DataFrame({
    'Task': ['Binary (label)', 'Multiclass (attack_cat)'],
    'Best model': [name_bin, name_multi],
    'Best params': [params_bin, params_multi],
    'Metric': ['F1', 'Macro F1'],
    'Score': [f1_binary, f1_multi]
})

display(summary.style.format({'Score': '{:.3f}'}).set_caption('Final results summary'))

### Final conclusions

This project explored intrusion detection through two complementary tasks: binary classification (normal vs. attack) and multiclass classification (attack category). For both tasks, the Random Forest algorithm, coupled with a preprocessing pipeline (VarianceThreshold + SelectKBest), proved to be the most effective strategy. In particular, the GridSearch validation dynamically selected 50 features for the binary task and 40 for the multiclass task out of the 185 original dataset features.

The binary model performed strongly (F1 = 0.922), with an error profile highly favorable for security applications. The multiclass task proved to be substantially harder (Macro F1 = 0.561), reflecting the overlap between several attack categories and the limited number of training examples available for the rarest classes.

Methodologically, PCA was explored as an alternative dimensionality reduction technique (notebook 03), but was excluded later on (notebook 04) due to impractical computation times during grid search. SMOTE was also considered as a way to address the multiclass imbalance, but was excluded for computational reasons.

Ultimately, this project demonstrates that traffic timing and byte-volume statistics are strong indicators of malicious network activity, though distinguishing between specific attack types proved challenging for rare categories. Furthermore, encapsulating the preprocessing steps within a Scikit-Learn Pipeline ensured zero data leakage during evaluation, guaranteeing robust and reliable results.